### 1 Try to extract features

In [2]:
import os
import glob
from bs4 import BeautifulSoup
import spacy
from tqdm.notebook import tqdm

# ==========================================
# 0. Initialization & Configuration (初始化与配置)
# ==========================================
# Load the NLP model. Unnecessary components are disabled to speed up NER.
# 加载NLP模型。禁用了语法解析等无关组件，专为NER提速。
# (If the books are in Chinese, use "zh_core_web_sm" instead of "en_core_web_sm")
# (如果小说是中文，请将 "en_core_web_sm" 替换为 "zh_core_web_sm")
nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "attribute_ruler", "lemmatizer"])
nlp.max_length = 5000000  # 增加这行，把字符上限扩大到 500 万

# Simulated Knowledge Base (A_quote / 模拟知识库)
# Replace with your actual A Song of Ice and Fire character/location list
# 请替换为你实际的《冰与火之歌》人物/地点列表
A_quote = {
    "Jon Snow", "Daenerys Targaryen", "Tyrion Lannister", "Arya Stark",
    "Sansa Stark", "Bran Stark", "Winterfell", "King's Landing", "The Wall",
    "琼恩·雪诺", "丹妮莉丝", "提利昂", "临冬城" 
}

# Target folder path (Relative path, pointing to your html folder)
# 目标文件夹路径 (相对路径，指向你截图中的 html 文件夹)
html_dir = "html"
html_files = glob.glob(os.path.join(html_dir, "*.html"))

# ==========================================
# 1. Core Processing Functions (定义核心处理函数)
# ==========================================
def parse_and_clean_html(file_path):
    """
    Read and process HTML, return clean text.
    读取并处理HTML，返回纯净的正文文本。
    """
    try:
        # Use errors='ignore' to prevent crashes from non-UTF8 legacy files
        # 使用 errors='ignore' 防止部分非utf-8编码的旧文件报错
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            html_content = f.read()
            
        soup = BeautifulSoup(html_content, 'lxml')
        
        # Remove noise tags (移除噪声标签)
        noise_tags = ['script', 'style', 'aside', 'nav', 'header', 'footer', 'meta']
        for tag in soup(noise_tags):
            tag.decompose()
            
        # Extract pure text blocks and join them (提取纯文本块并拼接)
        text_blocks = soup.stripped_strings
        return ' '.join(text_blocks)
    except Exception as e:
        print(f"Failed to parse (解析失败) {file_path}: {e}")
        return ""

def get_html_texts_generator(files):
    """
    Create a generator yielding cleaned HTML text and its filename.
    创建一个生成器，逐个产出清理后的HTML文本和对应文件名。
    """
    for file_path in files:
        clean_text = parse_and_clean_html(file_path)
        file_name = os.path.basename(file_path)
        yield clean_text, {"file_name": file_name}

def match_kb(extracted_entities, kb):
    """
    Knowledge Base Matching: Complete candidate retrieval.
    知识库匹配：完成候选召回。
    """
    matched = set()
    for ext_ent in extracted_entities:
        for kb_ent in kb:
            # Simple substring matching (简单的包含匹配)
            # Example: Extracts "Jon", matches with "Jon Snow"
            # 例如提取出 "Jon"，召回 "Jon Snow"
            if ext_ent in kb_ent or kb_ent in ext_ent:
                matched.add(kb_ent)
    return list(matched)

# ==========================================
# 2. Batch Execution Pipeline (批量执行 Pipeline)
# ==========================================
print(f"Starting process... Found {len(html_files)} HTML files. (开始处理，共找到 {len(html_files)} 个 HTML 文件...)")

# Store final results for all files (存储最终所有文件的结果)
results = {}

# Use nlp.pipe to efficiently process text stream 
# 使用 nlp.pipe 高效处理文本流 (as_tuples=True 允许传入附带文件名的元组)
text_stream = get_html_texts_generator(html_files)
pipeline = nlp.pipe(text_stream, as_tuples=True, batch_size=50)

# Iterate through results with tqdm progress bar
# 遍历处理结果，加入 tqdm 进度条
for doc, context in tqdm(pipeline, total=len(html_files), desc="NER & Retrieval Progress (NER & 召回进度)"):
    file_name = context["file_name"]
    
    # a. Extract candidate entities from spaCy Doc (List of C)
    # 从 spaCy Doc 中提取候选实体 (List of C)
    # Filter for Person, Location/Geo, Organization (限制为人名, 地理位置, 组织)
    list_of_C = set()
    for ent in doc.ents:
        if ent.label_ in {"PERSON", "GPE", "LOC", "ORG"}:
            list_of_C.add(ent.text)
            
    # b. Match with Knowledge Base (A_i)
    # 知识库匹配，完成召回 (A_i)
    A_i = match_kb(list(list_of_C), A_quote)
    
    # c. Save results (保存结果)
    results[file_name] = {
        "extracted_candidates": list(list_of_C),
        "recalled_entities": A_i
    }

print("Processing complete! All retrieved data is stored in the 'results' dictionary.")
print("处理完毕！所有召回数据已存储在 'results' 字典中。")

Starting process... Found 3594 HTML files. (开始处理，共找到 3594 个 HTML 文件...)


NER & Retrieval Progress (NER & 召回进度):   0%|          | 0/3594 [00:00<?, ?it/s]

Processing complete! All retrieved data is stored in the 'results' dictionary.
处理完毕！所有召回数据已存储在 'results' 字典中。


In [2]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     -- ------------------------------------- 0.8/12.8 MB 2.1 MB/s eta 0:00:06
     ----- ---------------------------------- 1.8/12.8 MB 3.2 MB/s eta 0:00:04
     ---------- ----------------------------- 3.4/12.8 MB 4.6 MB/s eta 0:00:03
     ------------------ --------------------- 6.0/12.8 MB 6.5 MB/s eta 0:00:02
     ------------------ --------------------- 6.0/12.8 MB 6.5 MB/s eta 0:00:02
     ----------------------------- ---------- 9.4/12.8 MB 6.8 MB/s eta 0:00:01
     ---------------------------------------  12.6/12.8 MB 8.2 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 7.9 MB/s  0:00:02
[+] Downl

In [3]:
!pip install lxml

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------- ----------------------------- 1.0/4.0 MB 6.3 MB/s eta 0:00:01
   --------------- ------------------------ 1.6/4.0 MB 4.4 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 7.3 MB/s  0:00:00


In [3]:
%%time
import os
import glob
import json
import re
from bs4 import BeautifulSoup
from tqdm import tqdm

class WikiFeatureExtractor:
    """网页特征提取器，用于提取实体的多维度特征"""
    
    def __init__(self, html_content):
        self.soup = BeautifulSoup(html_content, 'html.parser')
        
    def extract_identity(self):
        """1. 身份标识类特征"""
        identity = {}
        # 提取标题 (通常在 <h1 id="firstHeading"> 中)
        title_tag = self.soup.find('h1', id='firstHeading')
        identity['title'] = title_tag.text.strip() if title_tag else None
        return identity

    def extract_infobox(self):
        """3. 信息框结构化特征 (假设类名为 infobox)"""
        infobox_data = {}
        infobox = self.soup.find('table', class_=re.compile(r'infobox'))
        if infobox:
            rows = infobox.find_all('tr')
            for row in rows:
                th = row.find('th')
                td = row.find('td')
                if th and td:
                    key = th.text.strip().lower()
                    value = td.text.strip()
                    infobox_data[key] = value
        infobox_data['infobox_fields_count'] = len(infobox_data)
        return infobox_data

    def extract_text_features(self):
        """5. 正文文本类特征"""
        # 移除无用标签
        for tag in self.soup(['script', 'style', 'nav', 'table']):
            tag.decompose()
            
        paragraphs = self.soup.find_all('p')
        text = ' '.join([p.text.strip() for p in paragraphs])
        
        return {
            'text_length': len(text),
            'word_count': len(text.split()),
            'paragraph_count': len(paragraphs)
        }

    def extract_links(self):
        """7. 链接类特征"""
        links = self.soup.find_all('a', href=True)
        out_links = [link['href'] for link in links if link['href'].startswith('http')]
        internal_links = [link['href'] for link in links if link['href'].startswith('/wiki/')]
        
        return {
            'out_links_count': len(out_links),
            'internal_links_count': len(internal_links)
        }

    def extract_all(self):
        """聚合所有特征"""
        return {
            "identity": self.extract_identity(),
            "infobox": self.extract_infobox(),
            "text": self.extract_text_features(),
            "links": self.extract_links()
        }

# ==========================================
# 批量处理测试
# ==========================================
html_dir = "html"
html_files = glob.glob(os.path.join(html_dir, "*.html"))
features_dict = {}

print(f"开始特征工程提取，共 {len(html_files)} 个文件...")

for file_path in tqdm(html_files[:2000], desc="Feature Extraction"):  # 先用前100个文件测试
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            html_content = f.read()
            
        extractor = WikiFeatureExtractor(html_content)
        file_name = os.path.basename(file_path)
        features_dict[file_name] = extractor.extract_all()
        
    except Exception as e:
        pass

# 保存特征结果
with open("step2_features.json", "w", encoding="utf-8") as f:
    json.dump(features_dict, f, ensure_ascii=False, indent=4)
print("特征提取完成（features are ok），已保存至（have been saved） step2_features.json")

开始特征工程提取，共 3594 个文件...


Feature Extraction: 100%|██████████████████████████████████████████████████████████| 2000/2000 [01:06<00:00, 30.14it/s]


特征提取完成，已保存至 step2_features.json
CPU times: total: 1min 6s
Wall time: 1min 6s


In [17]:
import json
from IPython.display import display, Markdown

# 1. 读取你刚刚真实跑出来的特征数据
with open("step2_features.json", "r", encoding="utf-8") as f:
    all_features = json.load(f)

# 2. 随便取出一个人物的数据，用来分析你到底提取了哪些"组"和"特征"
sample_data = list(all_features.values())[0]

# 3. 动态生成 Markdown 格式的表格字符串
md_text = "### feature extraction (but it is so little)\n\n"
md_text += "Each group targets a specific part of the HTML page, automatically extracted from my local dataset.\n\n"
md_text += "| Group | Features |\n"
md_text += "| :--- | :--- |\n"

# 遍历你提取的每一组 (例如 Identity, Infobox)
for group_name, group_content in sample_data.items():
    if isinstance(group_content, dict):
        # 这一行非常重要，用来提取特征名并加上反引号
        features_list = [f"`{feature}`" for feature in group_content.keys()]
        # 把列表拼接成字符串
        features_str = ", ".join(features_list)
        # 把这一行加入表格
        md_text += f"| **{group_name}** | {features_str} |\n"

# 4. 在 Jupyter 中完美渲染展示出来！
display(Markdown(md_text))

### feature extraction (but it is so little)

Each group targets a specific part of the HTML page, automatically extracted from my local dataset.

| Group | Features |
| :--- | :--- |
| **identity** | `title` |
| **infobox** | `serabelar hightower`, `title`, `allegiance`, `culture`, `born`, `book`, `played by`, `tv series`, `infobox_fields_count` |
| **text** | `text_length`, `word_count`, `paragraph_count` |
| **links** | `out_links_count`, `internal_links_count` |


### 2 More complex feature need to make，need to make extended features



In [4]:
import os
import glob
import json
import re
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

class FullWikiFeatureExtractor:
    def __init__(self, html_content, file_path):
        self.soup = BeautifulSoup(html_content, 'html.parser')
        self.html_content = html_content
        self.file_path = file_path
        self.features = {}

    def extract_all(self):
        # 1. Identity (身份特征)
        title_tag = self.soup.find('h1', id='firstHeading')
        og_title = self.soup.find('meta', property='og:title')
        meta_desc = self.soup.find('meta', attrs={'name': 'description'})
        self.features['Identity'] = {
            'title': title_tag.text.strip() if title_tag else None,
            'og_title': og_title['content'] if og_title else None,
            'meta_description': meta_desc['content'] if meta_desc else None,
            'article_id': os.path.basename(self.file_path).replace('.html', '')
        }

        # 2. Infobox (信息框结构化数据) & Names
        infobox = self.soup.find('table', class_=re.compile(r'infobox'))
        infobox_data = {}
        if infobox:
            for row in infobox.find_all('tr'):
                th = row.find('th')
                td = row.find('td')
                if th and td:
                    key = th.text.strip().lower().replace(" ", "_")
                    infobox_data[key] = td.text.strip()
        
        self.features['Infobox'] = {
            'allegiances': infobox_data.get('allegiance', None),
            'culture': infobox_data.get('culture', None),
            'born': infobox_data.get('born', None),
            'died': infobox_data.get('died', None),
            'spouses': infobox_data.get('spouse(s)', None),
            'infobox_fields_count': len(infobox_data)
        }
        
        self.features['Names'] = {
            'aliases': infobox_data.get('alias', None),
            'all_name_variants': infobox_data.get('full_name', None)
        }

        # 3. Text (正文内容)
        paragraphs = self.soup.find_all('p')
        text_content = ' '.join([p.text.strip() for p in paragraphs])
        self.features['Text'] = {
            'text_length': len(text_content),
            'word_count': len(text_content.split()),
            'paragraph_count': len(paragraphs)
        }

        # 4. Structure (页面结构)
        sections = self.soup.find_all('span', class_='mw-headline')
        self.features['Structure'] = {
            'sections': [s.text.strip() for s in sections],
            'section_count': len(sections)
        }

        # 5. Links (链接特征)
        all_links = self.soup.find_all('a', href=True)
        out_links = [a['href'] for a in all_links if a['href'].startswith('http')]
        self.features['Links'] = {
            'links_out_count': len(out_links)
        }

        # 6. Categories (分类标签)
        cat_div = self.soup.find('div', id='mw-normal-catlinks')
        categories = []
        if cat_div:
            categories = [a.text for a in cat_div.find_all('a') if a.text != 'Categories']
        self.features['Categories'] = {
            'categories': categories,
            'category_count': len(categories)
        }

        # 7. Images (图片)
        images = self.soup.find_all('img')
        self.features['Images'] = {
            'has_portrait': bool(infobox and infobox.find('img')),
            'gallery_image_count': len(images)
        }

        # 8. Page (全局级别)
        self.features['Page'] = {
            'page_size_bytes': len(self.html_content.encode('utf-8'))
        }

        return self.features

# ==========================================
# 批量执行 4000 个文件的特征提取
# ==========================================
html_dir = "html"
html_files = glob.glob(os.path.join(html_dir, "*.html"))
all_features = {}

print(f"🚀 开始全量提取 11 大类特征，共 {len(html_files)} 个文件 (这可能需要几分钟)...")

for file_path in tqdm(html_files, desc="Extracted Pages"):
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        
        # 实例化提取器并运行
        extractor = FullWikiFeatureExtractor(content, file_path)
        all_features[os.path.basename(file_path)] = extractor.extract_all()
        
    except Exception as e:
        print(f"提取 {file_path} 时出错: {e}")

# 将最终的超级特征字典保存为 JSON
output_file = "step2_extended_features.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_features, f, ensure_ascii=False, indent=4)

print(f"✅ 完美！全量特征已拉取并保存至 {output_file}") 

🚀 开始全量提取 11 大类特征，共 3594 个文件 (这可能需要几分钟)...


Extracted Pages:   0%|          | 0/3594 [00:00<?, ?it/s]

✅ 完美！全量特征已拉取并保存至 step2_extended_features.json


In [10]:
import json
import random

# 加载你刚刚跑出来的特征数据
with open("step2_extended_features.json", "r", encoding="utf-8") as f:
    all_features = json.load(f)

# 过滤掉提取失败的空数据，随机挑一个角色展示
valid_files = [f for f, data in all_features.items() if data['Identity'].get('title')]
sample_file = random.choice(valid_files)
sample_data = all_features[sample_file]

print(f"🎯 随机抽取角色特征展示: {sample_file}")
print("=" * 50)
print(f"👤 【身份与名称】")
print(f" - 网页标题 (Title): {sample_data['Identity']['title']}")
print(f" - ID (Article ID): {sample_data['Identity']['article_id']}")
print(f" - 别名 (Aliases): {sample_data['Names']['aliases']}")

print(f"\n🛡️ 【信息框 (Infobox)】")
print(f" - 阵营 (Allegiances): {sample_data['Infobox']['allegiances']}")
print(f" - 文化 (Culture): {sample_data['Infobox']['culture']}")
print(f" - 出生 (Born): {sample_data['Infobox']['born']}")

print(f"\n📖 【文本与结构】")
print(f" - 正文词数 (Word Count): {sample_data['Text']['word_count']}")
print(f" - 段落数 (Paragraphs): {sample_data['Text']['paragraph_count']}")
print(f" - 提取到的分类数量 (Category count): {sample_data['Categories']['category_count']}")
print(f" - 是否有肖像图 (Has Portrait): {sample_data['Images']['has_portrait']}")
print("=" * 50)
print("💡 结论: 纯文本 HTML 已成功结构化为多维特征字典，随时可送入分类/排序模型！")

🎯 随机抽取角色特征展示: Rognar_II_Greyiron.html
👤 【身份与名称】
 - 网页标题 (Title): Rognar II Greyiron
 - ID (Article ID): Rognar_II_Greyiron
 - 别名 (Aliases): None

🛡️ 【信息框 (Infobox)】
 - 阵营 (Allegiances): None
 - 文化 (Culture): ironborn
 - 出生 (Born): None

📖 【文本与结构】
 - 正文词数 (Word Count): 60
 - 段落数 (Paragraphs): 3
 - 提取到的分类数量 (Category count): 5
 - 是否有肖像图 (Has Portrait): True
💡 结论: 纯文本 HTML 已成功结构化为多维特征字典，随时可送入分类/排序模型！


In [11]:
import json

# 读取刚刚保存的特征文件
with open("step2_extended_features.json", "r", encoding="utf-8") as f:
    extracted_data = json.load(f)

# 1. 统计总共提取了多少个网页
total_extracted = len(extracted_data)

# 2. 统计其中有多少个网页成功提取到了“标题（Title）”
valid_titles = sum(1 for data in extracted_data.values() if data['Identity'].get('title'))

# 3. 统计有多少个网页成功提取到了右侧的“信息框（Infobox）”
valid_infoboxes = sum(1 for data in extracted_data.values() if data['Infobox'].get('infobox_fields_count', 0) > 0)

print("📊 【提取数量统计报告】")
print("=" * 40)
print(f"📂 总计处理/提取的 HTML 文件数: {total_extracted} 个")
print(f"✅ 成功提取到核心标题(Title)的数量: {valid_titles} 个")
print(f"📦 成功解析出结构化信息框(Infobox)的数量: {valid_infoboxes} 个")
print("=" * 40)

📊 【提取数量统计报告】
📂 总计处理/提取的 HTML 文件数: 3594 个
✅ 成功提取到核心标题(Title)的数量: 3594 个
📦 成功解析出结构化信息框(Infobox)的数量: 3583 个


In [14]:
import json
from IPython.display import display, Markdown

# 1. 读取你刚刚真实跑出来的特征数据
with open("step2_extended_features.json", "r", encoding="utf-8") as f:
    all_features = json.load(f)

# 2. 随便取出一个人物的数据，用来分析你到底提取了哪些"组"和"特征"
sample_data = list(all_features.values())[0]

# 3. 动态生成 Markdown 格式的表格字符串
md_text = "### Extended feature extraction (My Actual Extracted Features)\n\n"
md_text += "Each group targets a specific part of the HTML page, automatically extracted from my local dataset.\n\n"
md_text += "| Group | Features |\n"
md_text += "| :--- | :--- |\n"

# 遍历你提取的每一组 (例如 Identity, Infobox)
for group_name, group_content in sample_data.items():
    if isinstance(group_content, dict):
        # 这一行非常重要，用来提取特征名并加上反引号
        features_list = [f"`{feature}`" for feature in group_content.keys()]
        # 把列表拼接成字符串
        features_str = ", ".join(features_list)
        # 把这一行加入表格
        md_text += f"| **{group_name}** | {features_str} |\n"

# 4. 在 Jupyter 中完美渲染展示出来！
display(Markdown(md_text))

### Extended feature extraction (My Actual Extracted Features)

Each group targets a specific part of the HTML page, automatically extracted from my local dataset.

| Group | Features |
| :--- | :--- |
| **Identity** | `title`, `og_title`, `meta_description`, `article_id` |
| **Infobox** | `allegiances`, `culture`, `born`, `died`, `spouses`, `infobox_fields_count` |
| **Names** | `aliases`, `all_name_variants` |
| **Text** | `text_length`, `word_count`, `paragraph_count` |
| **Structure** | `sections`, `section_count` |
| **Links** | `links_out_count` |
| **Categories** | `categories`, `category_count` |
| **Images** | `has_portrait`, `gallery_image_count` |
| **Page** | `page_size_bytes` |


### 3 check teacher's pkl，and change my josn to pkl

In [16]:
import pickle
import warnings

# 1. 加载数据
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=DeprecationWarning)
    with open("awoif_heavy.pkl", "rb") as f:
        df = pickle.load(f)

# ---------------------------------------------------------
# 2. 核心查看命令 (这三行最常用)
# ---------------------------------------------------------

print(f"数据形状 (行数, 列数): {df.shape}")
print("\n" + "-" * 50 + "\n")

print("列名 (提取的特征字段):")
print(df.columns.tolist())
print("\n" + "-" * 50 + "\n")

print("前 5 行数据预览:")
display(df.head()) # 如果是纯 Python 脚本，用 print(df.head())

数据形状 (行数, 列数): (3669, 10)

--------------------------------------------------

列名 (提取的特征字段):
['page', 'title', 'infobox_name', 'infobox', 'aliases_names', 'text_length', 'books', 'text', 'links', 'infobox_length']

--------------------------------------------------

前 5 行数据预览:


,page,title,infobox_name,infobox,aliases_names,text_length,books,text,links,infobox_length
0,Garth_the_Gardener,Garth the Gardener,Garth the Gardener,"{'reign': 'In the Age of Heroes', 'full name'...",[],513,"{'The World of Ice & Fire': 'mentioned', 'The ...",garth gardener legendary king reach founder ho...,"[Uthor_of_the_High_Tower, Brandon_of_the_Blood...",9
1,Nymor_Martell,Nymor Martell,Nymor Martell,"{'titles': ['Prince of Dorne', 'Lord of Sunspe...",[],2020,"{'The World of Ice & Fire': 'mentioned', 'Fire...",nymor martell ruling prince dorne head house m...,"[Maron_Martell, Princess_of_Dorne_(mother_of_D...",10
2,Lyman_Lannister,Lyman Lannister,Lyman Lannister,"{'titles': ['Lord of Casterly Rock', 'Shield o...",[],4402,"{'The World of Ice & Fire': 'mentioned', 'Fire...",lyman lannister lord casterly rock warden west...,"[Cerelle_Lannister_(daughter_of_Tybolt), Andro...",9
3,Tommen_Costayne_(knight),Tommen Costayne (knight),Tommen Costayne,"{'allegiance': 'House Costayne', 'culture': 'R...",[],361,{'The Hedge Knight': 'mentioned'},tommen costayne knight house costayne reign ki...,"[Daeron_II_Targaryen, Harlan_Grandison, Tom_Co...",5
4,Ernest_Dabell,Ernest Dabell,Ernest Dabell,"{'culture': 'Westeros', 'personal arms': 'A be...",[],325,{'The Hedge Knight': 'mentioned'},ser ernest dabell knight reign king daeron tar...,"[Daeron_II_Targaryen, Pascal_Dabell, Ernest_Da...",5


In [21]:
import pandas as pd
import json

# 1. 读取 JSON 数据
with open("step2_extended_features.json", "r", encoding="utf-8") as f:
    raw_dict = json.load(f)

# 2. 数据预处理：把文件名作为 'page' 列，合并到数据中
processed_list = []
for filename, features in raw_dict.items():
    # 创建一个新字典，把文件名放进去，然后展开特征
    row = {"page": filename}
    row.update(features)
    processed_list.append(row)

# 3. 关键一步：使用 json_normalize 自动扁平化所有嵌套字典
df = pd.json_normalize(processed_list, sep="_")

# 4. 过滤掉你不想要的列 (Images 和 Page 开头的)
# 找出所有需要删除的列名
cols_to_drop = [col for col in df.columns if col.startswith("Images_") or col.startswith("Page_")]
df = df.drop(columns=cols_to_drop)

# 5. 整理一下：把 '.html' 从 page 列去掉可选，看着更干净
df['page'] = df['page'].str.replace('.html', '', regex=False)

# ==========================================
# 查看结果并保存
# ==========================================

print(f" 转换完成！")
print(f"数据形状: {df.shape} (行数, 列数)")
print(f"\n最终列名 ({len(df.columns)} 列):")
print(df.columns.tolist())

print("\n" + "="*80)
print("前 3 行数据预览:")
display(df.head(3))

# 保存为 PKL 文件 (和你之前看的那个格式完全一样)
df.to_pickle("awoif_heavy2.pkl")
print("\n saved as awoif_heavy2.pkl")

 转换完成！
数据形状: (3594, 21) (行数, 列数)

最终列名 (21 列):
['page', 'Identity_title', 'Identity_og_title', 'Identity_meta_description', 'Identity_article_id', 'Infobox_allegiances', 'Infobox_culture', 'Infobox_born', 'Infobox_died', 'Infobox_spouses', 'Infobox_infobox_fields_count', 'Names_aliases', 'Names_all_name_variants', 'Text_text_length', 'Text_word_count', 'Text_paragraph_count', 'Structure_sections', 'Structure_section_count', 'Links_links_out_count', 'Categories_categories', 'Categories_category_count']

前 3 行数据预览:


,page,Identity_title,Identity_og_title,Identity_meta_description,Identity_article_id,Infobox_allegiances,Infobox_culture,Infobox_born,Infobox_died,Infobox_spouses,...,Names_aliases,Names_all_name_variants,Text_text_length,Text_word_count,Text_paragraph_count,Structure_sections,Structure_section_count,Links_links_out_count,Categories_categories,Categories_category_count
0,Abelar_Hightower,Abelar Hightower,Abelar Hightower,Abelar Hightower was a knight from House Hight...,Abelar_Hightower,House Hightower,Reach,"the Hightower[1], Oldtown",NaN,None,...,NaN,NaN,983,165,4,"[Appearance and Character, History, References]",3,10,"[House Hightower, 2nd century AC births, 3rd c...",6
1,Abelon,Abelon,Abelon,Abelon was an archmaester of the Citadel and t...,Abelon,The Citadel,Westerosi,NaN,NaN,None,...,NaN,NaN,104,18,1,[References],1,10,"[Archmaesters, Characters from Westeros, Scribes]",3
2,Addam_Frey,Addam Frey,Addam Frey,Addam Frey was a knight from House Frey during...,Addam_Frey,House Frey,Rivermen,the Twins[1],NaN,None,...,NaN,NaN,618,107,5,"[History, Family, References]",3,10,"[House Frey, 2nd century AC births, 3rd centur...",6



 已保存为 awoif_heavy2.pkl


### Look at the first two cells，Obviously, the features I extracted are richer compared to the simple version given to us by the teacher (but I know the teacher has a version with 40 features, while I only have more than 20).